# Mistério em SQL City

Adaptação do lendário [SQL Murder Mystery](https://github.com/NUKnightLab/sql-mysteries) (Knight Lab / Northwestern University, conteúdo original sob licença CC BY-SA 4.0) — mesmo caso, mesmos dados, mesmas pistas, só que sem SQL: aqui você resolve tudo com pandas + MinIO.

Um assassinato foi registrado em **SQL City** em **15/01/2018**. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar — sem nenhum pipeline pronto por trás, dessa vez é você quem constrói.

**Ferramentas: só pandas + MinIO + Jupyter.** Tudo se resolve com pandas puro: `pd.read_csv(...)` pra ler o CSV cru, `df.to_parquet(...)` pra publicar uma tabela no MinIO, `pd.read_parquet(...)` pra ler de volta. Ilustrado com exemplo logo na Fase 1, abaixo. Filtros, joins e agregações são sempre pandas puro (`merge`, `groupby`, filtro booleano...).

## O que você recebeu

6 CSVs em `dados/` — são as tabelas originais do banco do SQL Murder Mystery, cheias de gente e registros que não têm nada a ver com o caso (até 10 mil pessoas, mais de 1200 ocorrências) — filtrar direito é parte do trabalho:

| Arquivo | Colunas | O que é |
|---|---|---|
| `ocorrencia.csv` | `date, type, description, city` | Boletins de ocorrência de várias cidades, datas e tipos de crime |
| `pessoa.csv` | `id, name, license_id, address_number, address_street_name, ssn` | Cadastro de pessoas |
| `cnh.csv` | `id, age, height, eye_color, hair_color, gender, plate_number, car_make, car_model` | CNH — carteira de motorista, carro e placa |
| `depoimento.csv` | `person_id, transcript` | Depoimentos, ligados a `pessoa` por `person_id` |
| `membro_academia.csv` | `id, person_id, name, membership_start_date, membership_status` | Matrículas da academia "Get Fit Now Gym" |
| `checkin_academia.csv` | `membership_id, check_in_date, check_in_time, check_out_time` | Check-ins de entrada/saída na academia |

As datas vêm como inteiro `AAAAMMDD` (ex: `20180115`) em todas as tabelas que têm data — repare que é número, não texto, então não dá pra comparar direto com uma string tipo `"2018-01-15"`.

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

Sem spoiler aqui — a história e as pistas estão só nos dados. Boa investigação!

In [ ]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Conexão com o MinIO (S3-compatível) — pandas usa isso direto via s3fs
BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet flat** na camada — `bronze/<nome_tabela>.parquet`, sem subpasta por tabela. Feito isso, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [ ]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [ ]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

In [ ]:
# TODO: repita o padrão da célula do exemplo para "pessoa.csv" -> bronze.pessoa
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_pessoa

In [ ]:
# TODO: repita o padrão para "cnh.csv" -> bronze.cnh
df_cnh = pd.read_csv("dados/cnh.csv")
df_cnh.to_parquet(f"s3://{BUCKET}/bronze/cnh.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_cnh

In [ ]:
# TODO: repita o padrão para "depoimento.csv" -> bronze.depoimento
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_depoimento

In [ ]:
# TODO: repita o padrão para "membro_academia.csv" -> bronze.membro_academia
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_membro_academia

In [ ]:
# TODO: repita o padrão para "checkin_academia.csv" -> bronze.checkin_academia
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_checkin_academia

Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de agora é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Ache a ocorrência (`type == "murder"`, `city == "SQL City"`, `date == 20180115`) e leia a `description` com atenção — ela dá pistas de **endereço** para achar testemunhas em `pessoa`.
2. Ache as testemunhas em `pessoa` e cruze com `depoimento` para ler o que cada uma contou.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` (início do `id` da matrícula + status do plano), a outra para `cnh` (um trecho da placa).
4. Filtre cada tabela pela pista correspondente. Sozinha, cada pista pode sobrar **mais de 1** candidato — o cruzamento das duas é que deve fechar em exatamente **1** pessoa.
5. (Bônus) Confirme em `checkin_academia` que o suspeito tem um check-in na academia na data que a segunda testemunha mencionou.

As duas tabelas de candidatos do passo 3 não compartilham um id direto — vai precisar de `pessoa` como ponte entre elas (ver Passo 4).

In [ ]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)
# pessoa = pd.read_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS)
# cnh = pd.read_parquet(f"s3://{BUCKET}/bronze/cnh.parquet", storage_options=STORAGE_OPTIONS)
# depoimento = pd.read_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS)
# membro_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS)
# checkin_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS)

ocorrencia

### Passo 1 — a ocorrência

Filtre `ocorrencia` para achar o assassinato em SQL City no dia 15/01/2018 (`type == "murder"`, `city == "SQL City"`, `date == 20180115`), e leia a `description` inteira (`print(...)` ajuda a não truncar o texto).

In [ ]:
# TODO: filtre ocorrencia por type == "murder", city == "SQL City" e date == 20180115
# e dê print() na coluna "description" da linha encontrada.


### Passo 2 — as testemunhas

A descrição da ocorrência aponta para 2 pessoas em `pessoa`, cada uma identificada por uma pista de **rua/endereço** diferente (uma pelo número mais alto numa rua, a outra pelo nome numa outra rua). Ache as duas, depois cruze o `id` delas com `depoimento.person_id` para ler o que cada uma contou.

In [ ]:
# TODO: ache as 2 testemunhas em `pessoa` (via as pistas de rua/endereço da descrição)
# e depois os depoimentos delas em `depoimento` (merge por person_id, ou filtro direto).


### Passo 3 — duas pistas, duas tabelas

Um depoimento descreve algo sobre o **início do `id` da matrícula e o status do plano** de alguém na academia — filtre `membro_academia` por isso. Sozinha, essa pista deve deixar **mais de 1** candidato (tudo bem, é assim mesmo).

O outro depoimento descreve algo sobre **um trecho da placa** de quem fugiu — filtre `cnh` por isso.

In [ ]:
# TODO: filtre `membro_academia` pela pista do 1o depoimento (início do id da matrícula + status do plano)
# candidatos_academia = ...


In [ ]:
# TODO: filtre `cnh` pela pista do 2o depoimento (trecho da placa)
# candidatos_placa = ...


### Passo 4 — cruzando as pistas

`candidatos_academia` tem `person_id` (o mesmo `id` de `pessoa`), mas `candidatos_placa` não se liga direto a ele — o elo é `pessoa.license_id`, que aponta pro `id` de `cnh`. Cruze `candidatos_academia` com `pessoa` (por `person_id`/`id`) pra pegar o `license_id`, depois cruze o resultado com `candidatos_placa` (por `license_id`/`id`) — deve sobrar exatamente **1** pessoa. Se sobrar mais de 1 (ou 0), revise os filtros dos passos anteriores.

In [ ]:
# TODO: cruze candidatos_academia -> pessoa (person_id/id) -> candidatos_placa (license_id/id)
# e confira que sobrou 1 só suspeito
# suspeito = ...


### Passo 5 (bônus) — confirmar com o check-in

Cruze `membro_academia` com `checkin_academia` (por `id`/`membership_id`) e confira que o suspeito tem um check-in em `20180109` (09/01/2018) — a data que a segunda testemunha mencionou no depoimento.

In [ ]:
# TODO (bônus): confirme o check-in do suspeito em 09/01/2018 (20180109)


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `cnh`) que fechou o caso |
| `pista_academia` | qual detalhe da matrícula (início do `id` + status do plano) bateu com o depoimento |
| `pista_veiculo` | qual trecho da placa bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [ ]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver
# df_resposta = pd.DataFrame({...})
# df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)


---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.